# Winning Bidders — Bronze to Silver

This notebook transforms raw winning-bidder records from the **bronze** layer into a curated **silver** Delta table for downstream reporting and analytics.

It works at the **winning bid level**, meaning each row represents a bidder winning a specific lot in an auction. The notebook standardizes bidder fields, parses auction dates, enriches the data with `auction_id`, and loads the cleaned records into `company_db.silver.winningbidders`.

## What this notebook is doing

This notebook provides a breakdown of winning bidder activity, including:
* which auction the winning bid belongs to
* the lot number and item name won
* hammer price, buyer's premium, and total price realized
* the bidder ID and bidder name tied to the winning lot
* bidder email and billing and shipping details
* commission amount and ingestion metadata

## Main transformations

1. Reads raw winning-bidder records from `company_db.bronze.winningbidders`.
2. Builds a cleaned `full_name` from first and last name.
3. Renames source columns to analytics-friendly field names such as `bidder_id`, `lot_number`, and `item_name`.
4. Splits the auction date range into `start_date` and `end_date`.
5. Casts the parsed date fields to proper Spark `DATE` values.
6. Joins to `company_db.silver.auction_sales_report` to bring in `auction_id`.
7. Creates the target Delta table in the silver layer.
8. Registers the transformed DataFrame as a temporary view.
9. Merges transformed records into silver so existing winning-bid rows are updated and new rows are inserted.

## Source and target

* **Source:** `company_db.bronze.winningbidders`
* **Target:** `company_db.silver.winningbidders`
* **Storage location:** `abfss://auction@company.dfs.core.windows.net/silver/winningbidders`

## Data dictionary

| Column | Meaning |
| --- | --- |
| `auction_id` | Surrogate auction identifier from the silver auction sales report table. |
| `auction_date` | Original auction date range text from the source system. |
| `start_date` | Parsed auction start date from `auction_date`. |
| `end_date` | Parsed auction end date from `auction_date`. |
| `lot_number` | Lot number won by the bidder. |
| `item_name` | Name of the item associated with the lot. |
| `hammer_price` | Final hammer price before premiums or additional fees. |
| `buyers_premium` | Buyer premium amount applied to the winning bid. |
| `price_realized` | Total realized amount for the winning lot. |
| `bidder_id` | Bidder number assigned to the winning bidder. |
| `first_name` | Bidder first name. |
| `last_name` | Bidder last name. |
| `full_name` | Proper-cased full name built from first and last name. |
| `email` | Email address associated with the bidder. |
| `datetime` | Timestamp associated with the winning-bid record in the source. |
| `commission_amount` | Commission amount tied to the winning lot. |
| `bidder_billing_company` | Billing company name for the bidder. |
| `bidder_billing_first_name` | Billing contact first name. |
| `bidder_billing_last_name` | Billing contact last name. |
| `bidder_billing_phone_number` | Billing contact phone number. |
| `bidder_billing_address_1` | Primary billing street address. |
| `bidder_billing_address_2` | Secondary billing address line. |
| `bidder_billing_city` | Billing city. |
| `bidder_billing_state_province` | Billing state or province. |
| `bidder_billing_zip_postal_code` | Billing ZIP or postal code. |
| `bidder_billing_country` | Billing country. |
| `bidder_shipping_company` | Shipping company name for the bidder. |
| `bidder_shipping_first_name` | Shipping contact first name. |
| `bidder_shipping_last_name` | Shipping contact last name. |
| `bidder_shipping_phone_number` | Shipping contact phone number. |
| `bidder_shipping_address_1` | Primary shipping street address. |
| `bidder_shipping_address_2` | Secondary shipping address line. |
| `bidder_shipping_city` | Shipping city. |
| `bidder_shipping_state_province` | Shipping state or province. |
| `bidder_shipping_zip_postal_code` | Shipping ZIP or postal code. |
| `bidder_shipping_country` | Shipping country. |
| `path` | Source file or ingestion path used for traceability. |
| `ingestion_timestamp` | Timestamp when the raw record was ingested. |
| `created_at` | Timestamp when the record was first inserted into silver. |
| `updated_at` | Timestamp when the record was last updated in silver. |

## Business interpretation

* A row represents one winning bidder for a specific auction lot.
* `hammer_price`, `buyers_premium`, and `price_realized` support auction revenue and pricing analysis.
* `auction_id`, `lot_number`, and `bidder_id` form the business key used for upserts.
* The silver table supports operational reporting, bidder analysis, lot-level win analysis, and downstream joins to auction and bidder datasets.

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
from src.transformations.winning_bidders import transform_winning_bidders

In [ ]:
# Read the raw winning bidders table from the bronze layer.
# This DataFrame is the source for all lot-level cleaning and business transformations.
winningbidders = spark.table(f'{catalog}.bronze.winningbidders').filter(F.col('batch_id')==batch_id)

In [ ]:
winningbidders = transform_winning_bidders(winningbidders)

In [ ]:
# Enrich with auction ID
auction_sales_report = (
    spark.table(f"{catalog}.silver.auction_sales_report")
    .filter(F.col("batch_id") == batch_id)
    .select("start_date", "auction_id")
)

winningbidders = winningbidders.join(
    auction_sales_report,
    on="start_date",
    how="left"
)

In [ ]:
# Enrich with registered users
registeredusers = (
    spark.table(f"{catalog}.silver.registeredusers")
    .filter(F.col("batch_id") == batch_id)
    .select("user_id", "bidder_id")
)

In [ ]:
winningbidders = winningbidders.join(
    registeredusers,
    on="bidder_id",
    how="left"
)

In [ ]:
# Enrich with auction lot information
auctionresults = (
    spark.table(f"{catalog}.silver.auctionresults")
    .filter(F.col("batch_id") == batch_id)
    .select("lot_id", "lot_number")
)

winningbidders = winningbidders.join(
    auctionresults,
    on="lot_number",
    how="left"
)

# Remove duplicate records
winningbidders = winningbidders.dropDuplicates(
    ["start_date", "lot_number"]
)

In [ ]:

# Create or replace the silver Delta table that stores curated winning-bidder records.
# This schema supports reporting on auction results, bidder details, and realized pricing.
spark.sql(
f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.winningbidders (
    auction_id INT,
    auction_date STRING,
    start_date DATE,
    end_date DATE,
    lot_id INT,
    lot_number INT,
    item_name STRING,
    hammer_price DOUBLE,
    buyers_premium DOUBLE,
    price_realized DOUBLE,
    user_id INT,
    bidder_id INT,
    first_name STRING,
    last_name STRING,
    full_name STRING,
    email STRING,
    datetime TIMESTAMP,
    commission_amount DOUBLE,
    bidder_billing_company STRING,
    bidder_billing_first_name STRING,
    bidder_billing_last_name STRING,
    bidder_billing_phone_number STRING,
    bidder_billing_address_1 STRING,
    bidder_billing_address_2 STRING,
    bidder_billing_city STRING,
    bidder_billing_state_province STRING,
    bidder_billing_zip_postal_code STRING,
    bidder_billing_country STRING,
    bidder_shipping_company STRING,
    bidder_shipping_first_name STRING,
    bidder_shipping_last_name STRING,
    bidder_shipping_phone_number STRING,
    bidder_shipping_address_1 STRING,
    bidder_shipping_address_2 STRING,
    bidder_shipping_city STRING,
    bidder_shipping_state_province STRING,
    bidder_shipping_zip_postal_code STRING,
    bidder_shipping_country STRING,
    path STRING,
    ingestion_timestamp TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    batch_id STRING
)
USING DELTA 
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/silver/winningbidders';""")

In [ ]:
# Register the transformed DataFrame as a temporary view so it can be used by the SQL MERGE statement.
winningbidders.createOrReplaceTempView('winningbidders_view')

In [ ]:

# Upsert transformed winning-bidder records into the silver winning bidders table.
# Existing rows are updated by auction, lot, and bidder, and new rows are inserted.

spark.sql(
f"""
MERGE INTO {catalog}.silver.winningbidders AS tgt
USING winningbidders_view AS src
ON tgt.auction_id = src.auction_id
AND tgt.lot_number = src.lot_number
AND tgt.bidder_id = src.bidder_id

WHEN MATCHED
AND tgt.batch_id >= src.batch_id 
THEN
UPDATE SET
    tgt.auction_id = src.auction_id,
    tgt.auction_date = src.auction_date,
    tgt.start_date = src.start_date,
    tgt.end_date = src.end_date,
    tgt.lot_id = src.lot_id,
    tgt.lot_number = src.lot_number,
    tgt.item_name = src.item_name,
    tgt.hammer_price = src.hammer_price,
    tgt.buyers_premium = src.buyers_premium,
    tgt.price_realized = src.price_realized,
    tgt.user_id = src.user_id,
    tgt.bidder_id = src.bidder_id,
    tgt.first_name = src.first_name,
    tgt.last_name = src.last_name,
    tgt.full_name = src.full_name,
    tgt.email = src.email,
    tgt.datetime = src.datetime,
    tgt.commission_amount = src.commission_amount,
    tgt.bidder_billing_company = src.bidder_billing_company,
    tgt.bidder_billing_first_name = src.bidder_billing_first_name,
    tgt.bidder_billing_last_name = src.bidder_billing_last_name,
    tgt.bidder_billing_phone_number = src.bidder_billing_phone_number,
    tgt.bidder_billing_address_1 = src.bidder_billing_address_1,
    tgt.bidder_billing_address_2 = src.bidder_billing_address_2,
    tgt.bidder_billing_city = src.bidder_billing_city,
    tgt.bidder_billing_state_province = src.bidder_billing_state_province,
    tgt.bidder_billing_zip_postal_code = src.bidder_billing_zip_postal_code,
    tgt.bidder_billing_country = src.bidder_billing_country,
    tgt.bidder_shipping_company = src.bidder_shipping_company,
    tgt.bidder_shipping_first_name = src.bidder_shipping_first_name,
    tgt.bidder_shipping_last_name = src.bidder_shipping_last_name,
    tgt.bidder_shipping_phone_number = src.bidder_shipping_phone_number,
    tgt.bidder_shipping_address_1 = src.bidder_shipping_address_1,
    tgt.bidder_shipping_address_2 = src.bidder_shipping_address_2,
    tgt.bidder_shipping_city = src.bidder_shipping_city,
    tgt.bidder_shipping_state_province = src.bidder_shipping_state_province,
    tgt.bidder_shipping_zip_postal_code = src.bidder_shipping_zip_postal_code,
    tgt.bidder_shipping_country = src.bidder_shipping_country,
    tgt.path = src.path,
    tgt.ingestion_timestamp = src.ingestion_timestamp,
    tgt.updated_at = CURRENT_TIMESTAMP(),
    tgt.batch_id = src.batch_id

WHEN NOT MATCHED THEN
INSERT (
    auction_id,
    auction_date,
    start_date,
    end_date,
    lot_id,
    lot_number,
    item_name,
    hammer_price,
    buyers_premium,
    price_realized,
    user_id,
    bidder_id,
    first_name,
    last_name,
    full_name,
    email,
    datetime,
    commission_amount,
    bidder_billing_company,
    bidder_billing_first_name,
    bidder_billing_last_name,
    bidder_billing_phone_number,
    bidder_billing_address_1,
    bidder_billing_address_2,
    bidder_billing_city,
    bidder_billing_state_province,
    bidder_billing_zip_postal_code,
    bidder_billing_country,
    bidder_shipping_company,
    bidder_shipping_first_name,
    bidder_shipping_last_name,
    bidder_shipping_phone_number,
    bidder_shipping_address_1,
    bidder_shipping_address_2,
    bidder_shipping_city,
    bidder_shipping_state_province,
    bidder_shipping_zip_postal_code,
    bidder_shipping_country,
    path,
    ingestion_timestamp,
    created_at,
    updated_at,
    batch_id
)
VALUES (
    src.auction_id,
    src.auction_date,
    src.start_date,
    src.end_date,
    src.lot_id,
    src.lot_number,
    src.item_name,
    src.hammer_price,
    src.buyers_premium,
    src.price_realized,
    src.user_id,
    src.bidder_id,
    src.first_name,
    src.last_name,
    src.full_name,
    src.email,
    src.datetime,
    src.commission_amount,
    src.bidder_billing_company,
    src.bidder_billing_first_name,
    src.bidder_billing_last_name,
    src.bidder_billing_phone_number,
    src.bidder_billing_address_1,
    src.bidder_billing_address_2,
    src.bidder_billing_city,
    src.bidder_billing_state_province,
    src.bidder_billing_zip_postal_code,
    src.bidder_billing_country,
    src.bidder_shipping_company,
    src.bidder_shipping_first_name,
    src.bidder_shipping_last_name,
    src.bidder_shipping_phone_number,
    src.bidder_shipping_address_1,
    src.bidder_shipping_address_2,
    src.bidder_shipping_city,
    src.bidder_shipping_state_province,
    src.bidder_shipping_zip_postal_code,
    src.bidder_shipping_country,
    src.path,
    src.ingestion_timestamp,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    src.batch_id
);""")

## Analyst notes

Use this table when you need winning-bidder analysis such as:
* winning bids by auction
* lot-level results by bidder
* realized pricing by lot or auction
* bidder billing and shipping geography analysis
* commission and buyer premium reporting
* downstream joins to auction, lot, or bidder dimensions

Because the notebook stores one row per winning bidder per lot, it is well suited for operational reporting, analyst review, and silver-to-gold aggregations.

## Business usage notes

This silver table is the curated winning-bidder layer for auction results.

Use it to answer questions such as:
* which bidders won lots in each auction
* how realized prices compare across auctions, lots, or bidders
* how bidder geography is distributed across billing or shipping countries
* which winning bidders can be matched to downstream bidder or auction activity by bidder ID or email

This table is best treated as a winning-bid fact table at the auction-lot-bidder level.